# Language Models Homework 01

The central topic is parallel teacher-forced training versus sequential autoregressive inference with the **same** decoder-only Transformer. A GPT model does not gain an encoder during training that is later removed for inference. All core implementations live only in `src/`; the notebook does not and should not redefine them.

# Section 0 — Environment and reproducibility

## Goal

Explore environment and reproducibility through real execution and measurable evidence.

## Concept

The model and all tensors are deliberately created on the CPU, with fixed Python, NumPy, and PyTorch seeds.

## Your task

Run the setup cell and inspect the actual device of every parameter and the example tensor.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.


In [4]:
%load_ext autoreload
%autoreload 2

import csv
import json
import os
import platform
import random
import subprocess
import sys
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch

from src.config import MiniGPTConfig, count_parameters, set_seed
from src.model import MiniGPT

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir() and (path / "assets").is_dir()
)
ASSETS_DIR = ROOT / "assets"
FIGURES_DIR = ROOT / "figures"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TESTS_DIR = ROOT / "tests"
if (TESTS_DIR / "public").is_dir():
    TESTS_DIR = TESTS_DIR / "public"

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if not rows:
        raise ValueError("CSV evidence must contain at least one row")
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

def run_public_test(filename):
    command = [sys.executable, "-m", "pytest", str(TESTS_DIR / filename), "-q"]
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(result.stdout + result.stderr)
    print(result.stdout.strip().splitlines()[-1])

SEED = 1403
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)
torch.set_num_threads(min(6, torch.get_num_threads()))
device = torch.device("cpu")
model = MiniGPT(MiniGPTConfig()).to(device)
example_tensor = torch.tensor([[1, 4, 5]], dtype=torch.long, device=device)
assert all(parameter.device.type == "cpu" for parameter in model.parameters())
assert example_tensor.device.type == "cpu"
assert count_parameters(model) == 644_736
environment = {
    "python": sys.version.split()[0],
    "pytorch": torch.__version__,
    "cpu": platform.processor() or platform.machine(),
    "logical_cpu_count": psutil.cpu_count(),
    "cuda_available_report_only": torch.cuda.is_available(),
    "forced_device": device.type,
    "parameter_count": count_parameters(model),
    "seed": SEED,
}
environment

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


{'python': '3.15.2',
 'pytorch': '2.15.0+cpu',
 'cpu': 'Intel64 Family 6 Model 186 Stepping 2, GenuineIntel',
 'logical_cpu_count': 16,
 'cuda_available_report_only': False,
 'forced_device': 'cpu',
 'parameter_count': 644736,
 'seed': 1403}

# Section 1 — Byte tokenization

## Goal

Explore byte tokenization through real execution and measurable evidence.

## Concept

Each UTF-8 byte is offset by four so that the special-token IDs remain available.

## Your task

Complete the TODOs in `src/tokenizer.py`; this notebook only imports and runs those functions.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [7]:
from src.tokenizer import decode_tokens, encode_text

tokenizer_examples = []
for text in ["Hello!", "café MiniGPT", ""]:
    token_ids = encode_text(text, add_bos=True, add_eos=True)
    decoded = decode_tokens(token_ids[1:-1])
    assert decoded == text
    tokenizer_examples.append(
        {"text": text, "token_ids": token_ids, "decoded_without_specials": decoded}
    )
write_json(RESULTS_DIR / "tokenizer_examples.json", tokenizer_examples)
run_public_test("test_tokenizer.py")
tokenizer_examples

8 passed in 1.80s


[{'text': 'Hello!',
  'token_ids': [1, 76, 105, 112, 112, 115, 37, 2],
  'decoded_without_specials': 'Hello!'},
 {'text': 'café MiniGPT',
  'token_ids': [1,
   103,
   101,
   106,
   199,
   173,
   36,
   81,
   109,
   114,
   109,
   75,
   84,
   88,
   2],
  'decoded_without_specials': 'café MiniGPT'},
 {'text': '', 'token_ids': [1, 2], 'decoded_without_specials': ''}]

# Section 2 — Input and label shifting

## Goal

Explore input and label shifting through real execution and measurable evidence.

## Concept

Each input position must predict the token at the next position, so inputs and labels are shifted by one token.

## Your task

Complete the `create_lm_example` TODO in `src/data.py`.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.


In [9]:
from src.data import create_lm_example
from src.tokenizer import token_display

shifted_ids = torch.tensor(encode_text("The cat sat", add_bos=True, add_eos=True), dtype=torch.long)
shifted_input, shifted_labels = create_lm_example(shifted_ids)
shift_rows = [
    {
        "position": position,
        "input_token_id": int(input_id),
        "input_token_display": token_display(int(input_id)),
        "target_token_id": int(target_id),
        "target_token_display": token_display(int(target_id)),
    }
    for position, (input_id, target_id) in enumerate(
        zip(shifted_input, shifted_labels, strict=True)
    )
]
write_csv(RESULTS_DIR / "token_shift_table.csv", shift_rows)
run_public_test("test_data.py")
pd.DataFrame(shift_rows).head()

3 passed in 2.04s


,position,input_token_id,input_token_display,target_token_id,target_token_display
0,0,1,<BOS>,88,T
1,1,88,T,108,h
2,2,108,h,105,e
3,3,105,e,36,
4,4,36,,103,c


# Section 3 — Causal attention mask

## Goal

Explore causal attention mask through real execution and measurable evidence.

## Concept

Position i may attend only to itself and the past. A controlled experiment changes future tokens and measures earlier logits.

## Your task

Complete the mask TODO in `src/masking.py`; the scaffold creates the figure and leakage experiment.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [ ]:
from src.masking import build_causal_mask
from src.plotting import plot_causal_mask

causal_mask = build_causal_mask(16, device=device)
plot_causal_mask(causal_mask, FIGURES_DIR / "causal_mask.png")

set_seed(SEED)
leakage_model = MiniGPT(
    MiniGPTConfig(
        context_length=8, n_layers=1, d_model=32, n_heads=4, d_ff=64, dropout=0.0
    )
).to(device).eval()
original = torch.tensor([[10, 11, 12, 13]], device=device)
changed_future = torch.tensor([[10, 11, 99, 98]], device=device)
with torch.inference_mode():
    masked_difference = float(
        (leakage_model(original).logits[:, :2] - leakage_model(changed_future).logits[:, :2])
        .abs()
        .max()
    )
    unmasked_difference = float(
        (
            leakage_model(original, attention_mask=False).logits[:, :2]
            - leakage_model(changed_future, attention_mask=False).logits[:, :2]
        )
        .abs()
        .max()
    )
assert masked_difference < 1e-6 and unmasked_difference > 1e-6
leakage_evidence = {
    "mask_shape": list(causal_mask.shape),
    "blocked_entries": int(causal_mask.sum()),
    "masked_early_logit_max_difference": masked_difference,
    "unmasked_early_logit_max_difference": unmasked_difference,
}
run_public_test("test_masking.py")
leakage_evidence

# Section 4 — Parallel training with teacher forcing

## Goal

Explore parallel training with teacher forcing through real execution and measurable evidence.

## Concept

One forward pass produces logits for all B×T known positions; loss, backward, and the optimizer step follow.

## Your task

Complete the `training_step` TODO in `src/training.py`.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [ ]:
from src.data import load_corpus_tokens, sample_batch
from src.training import training_step

set_seed(SEED)
step_model = MiniGPT(MiniGPTConfig()).to(device)
optimizer = torch.optim.AdamW(step_model.parameters(), lr=1e-3)
corpus_tokens = load_corpus_tokens(ASSETS_DIR / "corpus.txt").to(device)
step_inputs, step_labels = sample_batch(
    corpus_tokens, 2, 32, torch.Generator(device="cpu").manual_seed(SEED)
)
step_inputs, step_labels = step_inputs.to(device), step_labels.to(device)
before = step_model.token_embedding.weight.detach().clone()
observed = {"forward_passes": 0}

def capture_forward(_module, _inputs, output):
    observed["forward_passes"] += 1
    observed["logit_shape"] = list(output.logits.shape)

hook = step_model.register_forward_hook(capture_forward)
step_metrics = training_step(step_model, step_inputs, step_labels, optimizer)
hook.remove()
step_metrics.update(
    {
        "batch_shape": list(step_inputs.shape),
        "logit_shape": observed["logit_shape"],
        "label_shape": list(step_labels.shape),
        "forward_passes": float(observed["forward_passes"]),
        "parameters_changed": not torch.equal(before, step_model.token_embedding.weight),
        "model_training_mode": step_model.training,
    }
)
assert observed["forward_passes"] == 1
write_json(RESULTS_DIR / "training_step_metrics.json", step_metrics)
run_public_test("test_training.py")
step_metrics

# Section 5 — Short CPU training experiment

## Goal

Explore short cpu training experiment through real execution and measurable evidence.

## Concept

This 20-step run demonstrates optimization mechanics; it does not train a strong language model.

## Your task

Use the fixed scaffold; the training-step algorithm is the TODO from the previous section.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [ ]:
from src.data import split_tokens
from src.plotting import plot_training_loss
from src.training import train_fixed_steps

set_seed(SEED)
short_model = MiniGPT(MiniGPTConfig()).to(device)
train_tokens, validation_tokens = split_tokens(corpus_tokens)
losses, training_metrics = train_fixed_steps(
    short_model,
    train_tokens,
    validation_tokens,
    steps=20,
    batch_size=4,
    sequence_length=64,
    learning_rate=8e-4,
    seed=SEED,
    validation_batches=4,
)
training_metrics["loss_history"] = losses
write_json(RESULTS_DIR / "training_metrics.json", training_metrics)
plot_training_loss(losses, FIGURES_DIR / "training_loss.png")
{key: value for key, value in training_metrics.items() if key != "loss_history"}

# Section 6 — Manual greedy autoregressive generation

## Goal

Explore manual greedy autoregressive generation through real execution and measurable evidence.

## Concept

At each step, only the logits at the final position are used, and the selected token becomes part of the next input.

## Your task

Complete the `greedy_generate` TODO in `src/generation.py`.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [ ]:
from src.generation import greedy_generate

checkpoint = torch.load(
    ASSETS_DIR / "minigpt_pretrained.pt", map_location=device, weights_only=True
)
reference_model = MiniGPT(MiniGPTConfig(**checkpoint["model_config"])).to(device)
reference_model.load_state_dict(checkpoint["model_state_dict"])
reference_model.eval()
assert all(parameter.device.type == "cpu" for parameter in reference_model.parameters())

prompt = 'Holmes looked at the letter and said, "'
prompt_ids = torch.tensor(encode_text(prompt, add_bos=True), dtype=torch.long, device=device)
generated_ids, generation_trace = greedy_generate(
    reference_model, prompt_ids, 80, 2, reference_model.config.context_length
)
write_csv(RESULTS_DIR / "generation_trace.csv", generation_trace)
generated_text = decode_tokens([token_id for token_id in generated_ids.tolist() if token_id >= 4])
run_public_test("test_generation.py")
{
    "generated_text": generated_text,
    "new_tokens": len(generation_trace),
    "forward_passes": generation_trace[-1]["forward_pass_count"],
}

# Section 7 — Teacher forcing versus free-running histories

## Goal

Explore teacher forcing versus free-running histories through real execution and measurable evidence.

## Concept

At row t, both distributions predict token x_t from the prefix before t. The incorrect token is injected after measuring that row, so divergence is expected from the following row.

## Your task

Complete the `total_variation` and `measure_error_propagation` TODOs in `src/analysis.py`.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [ ]:
from src.analysis import measure_error_propagation
from src.plotting import plot_error_propagation

reference_ids = torch.tensor(
    encode_text("The detective opened the door and looked into the quiet room."),
    dtype=torch.long,
    device=device,
)[:48]
injection_position = 12
error_rows = measure_error_propagation(
    reference_model, reference_ids, injection_position=injection_position
)
write_csv(RESULTS_DIR / "error_propagation.csv", error_rows)
plot_error_propagation(error_rows, FIGURES_DIR / "error_propagation.png")
assert all(row["total_variation"] < 1e-6 for row in error_rows if row["phase"] != "after")
assert any(row["total_variation"] > 1e-6 for row in error_rows if row["phase"] == "after")
run_public_test("test_analysis.py")
pd.DataFrame(error_rows).loc[:, ["position", "phase", "total_variation"]].head(16)

# Section 8 — Training and inference comparison

## Goal

Explore training and inference comparison through real execution and measurable evidence.

## Concept

Both paths use the same decoder-only MiniGPT architecture. `eval()` controls module behavior, while `inference_mode()` controls autograd recording.

## Your task

Run the comparison table and four-case mode/gradient experiment with the scaffold.

## Constraints

Implement algorithms only in `src/`. Do not redefine a function with the same name in the notebook. You may not use `generate()`, a pretrained external model, a GPU, an external API, or fixed outputs.

## Hint

After you edit `src/`, autoreload imports the updated code. Pay attention to tensor shapes, conditioning prefixes, and the number of forward passes.

In [ ]:
comparison_rows = [
    {"dimension": "architecture", "training": "same decoder-only MiniGPT", "inference": "same decoder-only MiniGPT"},
    {"dimension": "model mode", "training": "train()", "inference": "eval()"},
    {"dimension": "ground-truth previous tokens", "training": "yes", "inference": "prompt only"},
    {"dimension": "self-generated previous tokens", "training": "no", "inference": "yes"},
    {"dimension": "forward passes", "training": "1 for the measured batch", "inference": str(len(generation_trace))},
    {"dimension": "logits used", "training": "all valid positions", "inference": "final position per step"},
    {"dimension": "gradient tracking", "training": "enabled", "inference": "disabled"},
    {"dimension": "optimizer", "training": "used", "inference": "not used"},
    {"dimension": "dropout", "training": "active", "inference": "disabled by eval()"},
    {"dimension": "known-position parallelism", "training": "B x T positions", "inference": "within each current prefix"},
    {"dimension": "decoding dependency", "training": "labels already known", "inference": "next input waits for selected token"},
    {"dimension": "memory", "training": "activations, gradients, optimizer", "inference": "no autograd graph or optimizer"},
]
write_csv(RESULTS_DIR / "training_vs_inference.csv", comparison_rows)

set_seed(SEED)
mode_model = MiniGPT(
    MiniGPTConfig(
        context_length=8, n_layers=1, d_model=32, n_heads=4, d_ff=64, dropout=0.5
    )
).to(device)
mode_input = torch.tensor([[4, 5, 6, 7]], dtype=torch.long, device=device)
mode_rows = []
configurations = [
    ("train + grad", True, False),
    ("eval + grad", False, False),
    ("train + inference_mode", True, True),
    ("eval + inference_mode", False, True),
]
for configuration, training_mode, inference in configurations:
    mode_model.train(training_mode)
    torch.manual_seed(2025)
    context = torch.inference_mode() if inference else nullcontext()
    with context:
        grad_enabled = torch.is_grad_enabled()
        first_output = mode_model(mode_input).logits
        second_output = mode_model(mode_input).logits
    repeat_equal = bool(torch.equal(first_output, second_output))
    mode_rows.append(
        {
            "configuration": configuration,
            "model.training": mode_model.training,
            "torch.is_grad_enabled": grad_enabled,
            "output.requires_grad": first_output.requires_grad,
            "dropout_active_or_repeat_outputs_equal": repeat_equal,
        }
    )
write_csv(RESULTS_DIR / "eval_vs_inference_mode.csv", mode_rows)
by_name = {row["configuration"]: row for row in mode_rows}
assert by_name["eval + grad"]["output.requires_grad"] is True
assert by_name["train + inference_mode"]["model.training"] is True
assert by_name["train + inference_mode"]["torch.is_grad_enabled"] is False
assert by_name["eval + grad"]["dropout_active_or_repeat_outputs_equal"] is True
assert by_name["train + grad"]["dropout_active_or_repeat_outputs_equal"] is False
pd.DataFrame(mode_rows)

# Final Validation and Packaging

The next cell validates every artifact and all public tests, then displays the packaging command. Set `HW01_BUILD_SUBMISSION=1` to run packaging automatically. Assets, model weights, caches, and virtual environments must not be included in the submission ZIP.

In [ ]:
from src.validation import validate_artifacts

artifact_summary = validate_artifacts(ROOT)
full_test_command = [sys.executable, "-m", "pytest", str(TESTS_DIR), "-q"]
full_test_result = subprocess.run(full_test_command, cwd=ROOT, text=True, capture_output=True)
if full_test_result.returncode:
    raise RuntimeError(full_test_result.stdout + full_test_result.stderr)
print(full_test_result.stdout.strip().splitlines()[-1])
package_command = "uv run python scripts/package_submission.py --student-id 123456"
if (ROOT / "scripts" / "package_submission.py").is_file() and os.environ.get("HW01_BUILD_SUBMISSION") == "1":
    subprocess.run(package_command.split(), cwd=ROOT, check=True)
else:
    print("Packaging command:", package_command)
print(f"Validated {len(artifact_summary)} required artifacts.")